# 00 · Reproduce Model B on CUAD (Milestone 0)

Runs `Ihteshamstar/qwen3-4b-cuad-extractor` with its native prompt on 10 of the 102 official
CUAD test contracts, at the model card's chunking (60k/8k chars) and at the 32k/4k chunking
planned for credit agreements. It reports detection F1 against the card's numbers, plus
loop/ladder stats, latency per chunk, and GPU minutes.

**Kaggle:** Accelerator **GPU T4 x2** (not P100), Internet **on**, Add-ons → Secrets: attach
`HF_TOKEN` (write access). "Save & Run All" runs it in the background.
**Colab:** T4 runtime, add `HF_TOKEN` under 🔑 Secrets with notebook access enabled.

Resumable: finished chunks are pulled from the private HF dataset repo and skipped.
All logic lives in `src/`; this notebook only wires it together.

In [ ]:
REPO_URL = "https://github.com/z26qin/credit-extractor.git"
BRANCH = "m0-reproduce"
INSTALL_VLLM = True      # vLLM goes into its own venv (/tmp/vllm-env); False -> transformers only

In [ ]:
import os, subprocess, sys
ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else (
    "/content" if os.path.exists("/content") else os.getcwd())
REPO = os.path.join(ROOT, "credit-extractor")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
# register the repo on sys.path permanently, so `import src` survives a kernel restart
import site
with open(os.path.join(site.getsitepackages()[0], "credit_extractor.pth"), "w") as f:
    f.write(REPO + "\n")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(pip + ["-r", "requirements.txt"], check=True)

# 1) platform Python (transformers backend): never install vLLM here. Its torch pin
#    breaks the preinstalled torchaudio, which transformers imports.
BASE_CHECK = ("import torch, transformers\n"
              "from transformers import Qwen3ForCausalLM\n"
              "print('base: torch', torch.__version__, 'cuda', torch.version.cuda,"
              " 'gpu', torch.cuda.is_available(), '| transformers', transformers.__version__)")
r = subprocess.run([sys.executable, "-c", BASE_CHECK], capture_output=True, text=True)
if r.returncode and "torchaudio" in r.stderr.lower():
    print("torchaudio does not match torch; removing it (not needed for text models)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"])
    r = subprocess.run([sys.executable, "-c", BASE_CHECK], capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])
assert r.returncode == 0, "base env broken: Run -> Factory reset, then run all cells again"

# 2) vLLM in its own venv; uv picks the torch build that matches this GPU driver
VENV = "/tmp/vllm-env"; VPY = VENV + "/bin/python"
VLLM_CHECK = ("import torch, vllm\n"
              "print('vllm venv: vllm', vllm.__version__, '| torch', torch.__version__,"
              " 'cuda', torch.version.cuda, 'gpu', torch.cuda.is_available())")
if INSTALL_VLLM and not os.path.exists(VPY):
    subprocess.run(pip + ["uv"], check=True)
    uv = [sys.executable, "-m", "uv"]
    subprocess.run(uv + ["venv", "-q", VENV, "--python", sys.executable], check=True)
    r = subprocess.run(uv + ["pip", "install", "-q", "--python", VPY, "--torch-backend=auto",
                             "-r", "requirements-gpu.txt", "-r", "requirements.txt", "wrapt"],
                       capture_output=True, text=True)
    if r.returncode:
        print("vllm install FAILED -> transformers only\n" + r.stderr[-2000:])
        import shutil; shutil.rmtree(VENV, ignore_errors=True)
if os.path.exists(VPY):
    r = subprocess.run([VPY, "-c", VLLM_CHECK], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else
          "vllm venv not usable (runner will fall back to transformers):\n" + r.stderr[-1500:])
else:
    print("no vllm venv: runner uses transformers")

In [ ]:
# after a kernel restart, start again from this cell (the cells below need `env`)
from src.platform import setup
env = setup()
assert env.data_repo, "HF_TOKEN missing: attach it as a secret (outputs must sync to the Hub)"
assert env.gpus, "no GPU visible: pick GPU T4 x2 (Kaggle) or a T4 runtime (Colab)"

In [ ]:
from src import reproduce
cfg = reproduce.load_cfg()
plan = reproduce.prepare(env, cfg)      # CUAD download, Hub pull, run dirs (fingerprinted)

In [ ]:
# primary greedy pass (vLLM TP=2 -> vLLM TP=1 -> transformers), then the
# repeat-penalty ladder (1.05 -> 1.15 -> 1.25, temperature 0) on failed chunks only
used = reproduce.run(env, plan, cfg)
used

In [ ]:
results = reproduce.report(env, plan, cfg)   # also writes report.md + results.json to the Hub

In [ ]:
import pandas as pd
for name, rd in plan["run_dirs"].items():
    df = pd.read_csv(rd / "latency.csv")
    print(name); display(df.sort_values(["chunk_id", "attempt"]))
print(f"session wall time: {env.session_minutes():.1f} min")